In [1]:
"""Tutorial: Electron Repulsion Integrals via Obara-Saika Recursion"""

__authors__   = ["Kien Bui"]
__credits__   = ["Kien Bui"]

__copyright__ = "(c) 2014-2018, The Psi4NumPy Developers"
__license__   = "BSD-3-Clause"
__date__      = "2026-08-10"

# Electron Repulsion Integrals

Tutorials [11c](11c_one_electron_integrals.ipynb) built every one-electron integral needed for Hartree-Fock. This tutorial completes the set with the two-electron electron repulsion integrals (ERIs). The good news is that there is essentially no new physics, as it's mostly similar to the nuclear attraction integrals. The bad news is bookkeeping: the ERI has four centers instead of two, meaning twelve spatial indices, so the working array becomes 13-dimensional.

Contents:

1. Definition, and the four-center geometry: $\mathbf P$, $\mathbf Q$, $\mathbf W$, and $\rho$
2. The recursion, and the thirteen-dimensional working array
3. Eight-fold permutational symmetry, and how to exploit it in the shell loop
4. Assembling the ERI tensor and checking against Psi4

## Definition

In chemists' notation, the two-electron repulsion integral over four primitives is

\begin{equation}
(\mathbf a \mathbf b | \mathbf c \mathbf d) =
\iint d\mathbf r_1\, d\mathbf r_2\;
\frac{\phi_a(\mathbf r_1)\phi_b(\mathbf r_1)\, \phi_c(\mathbf r_2)\phi_d(\mathbf r_2)}
{|\mathbf r_1 - \mathbf r_2|}
\tag{[Obara:1986] Eqn 23}
\end{equation}

Functions $\mathbf a$ and $\mathbf b$ describe electron 1, functions $\mathbf c$ and $\mathbf d$ describe electron 2, and the operator couples them.

## The four-center geometry

We have to apply the Gaussian product theorem twice, once to each electron's pair. Each pair collapses to a single Gaussian on its own weighted center:

\begin{equation}
\zeta = a + b, \quad \mathbf P = \frac{a\mathbf A + b\mathbf B}{\zeta};
\qquad
\eta = c + d, \quad \mathbf Q = \frac{c\mathbf C + d\mathbf D}{\eta}
\end{equation}

The two product Gaussians, $\mathbf P$ and $\mathbf Q$, then couple through a combined center $\mathbf W$ and a reduced exponent $\rho$:

\begin{equation}
\mathbf W = \frac{\zeta\mathbf P + \eta\mathbf Q}{\zeta + \eta},
\qquad
\rho = \frac{\zeta\eta}{\zeta + \eta}
\end{equation}

The electrion repulsion variables mirror those of the nuclear attraction, and seeing the connection makes the code below easier to read:

| Nuclear attraction (11c) | Electron repulsion (here) |
|---|---|
| $\mathbf P - \mathbf A$, $\mathbf P - \mathbf B$ | $\mathbf P - \mathbf A$, $\mathbf P - \mathbf B$ (bra), $\mathbf Q - \mathbf C$, $\mathbf Q - \mathbf D$ (ket) |
| $\mathbf P - \mathbf C$ (nucleus) | $\mathbf W - \mathbf P$ (bra), $\mathbf W - \mathbf Q$ (ket) |
| $U = \zeta\lvert\mathbf P - \mathbf C\rvert^2$ | $T = \rho\lvert\mathbf P - \mathbf Q\rvert^2$ |
| $2\sqrt{\zeta/\pi}\; S_{\text{base}}$ | $2\sqrt{\rho/\pi}\; S^{(ab)}_{\text{base}} S^{(cd)}_{\text{base}}$ |

## The seed and the prefactor

Similarly as in the NAI, we seed on the Boys function but with the argument $T = \rho|\mathbf P - \mathbf Q|^2$:

\begin{equation}
[\mathbf{00}|\mathbf{00}]^{(m)} = F_m(T),
\qquad m = 0, \ldots, \ell_a + \ell_b + \ell_c + \ell_d
\end{equation}

and the prefactor is the product of both pairs' Gaussian prefactors:

\begin{equation}
\text{ERI}_{\text{base}} = 2\sqrt{\frac{\rho}{\pi}}\;
S^{(ab)}_{\text{base}} S^{(cd)}_{\text{base}}
\tag{[Obara:1986] Eqn 44}
\end{equation}

As before, only $m = 0$ is physically meaningful; the higher orders are scaffolding for the climb in angular momentum, and the code reads out only the `[..., 0]` slice.

## The recursion

Stepping up the $x$ index on center A:

\begin{equation}
[\mathbf a \,\mathbf b |\mathbf c \,\mathbf d ]^{(m)} =
(P_x - A_x)\, [\mathbf a{-}\mathbf 1_x\,\mathbf b |\mathbf c \,\mathbf d ]^{(m)}
+ (W_x - P_x)\, [\mathbf a{-}\mathbf 1_x\,\mathbf b |\mathbf c \,\mathbf d ]^{(m+1)}
\end{equation}

\begin{equation}
+ \frac{a_x - 1}{2\zeta}\Big([\mathbf a{-}\mathbf 2_x\,\mathbf b |\mathbf c \,\mathbf d ]^{(m)} - \tfrac{\eta}{\zeta+\eta}[\mathbf a{-}\mathbf 2_x\,\mathbf b |\mathbf c \,\mathbf d ]^{(m+1)}\Big)
\end{equation}

\begin{equation}
+ \frac{b_x}{2\zeta}\Big([\mathbf a{-}\mathbf 1_x\,\mathbf b{-}\mathbf 1_x|\mathbf c \,\mathbf d ]^{(m)} - \tfrac{\eta}{\zeta+\eta}[\mathbf a{-}\mathbf 1_x\,\mathbf b{-}\mathbf 1_x|\mathbf c \,\mathbf d ]^{(m+1)}\Big)
\end{equation}

\begin{equation}
+ \frac{c_x}{2(\zeta+\eta)}[\mathbf a{-}\mathbf 1_x\,\mathbf b |\mathbf c{-}\mathbf 1_x\,\mathbf d ]^{(m+1)}
+ \frac{d_x}{2(\zeta+\eta)}[\mathbf a{-}\mathbf 1_x\,\mathbf b |\mathbf c \,\mathbf d{-}\mathbf 1_x]^{(m+1)}
\end{equation}

Consistent with 11c, the notation here is written in **step-down** form rather than the step-up form of the original paper. This indexing reads more naturally in the code as we are looping over indices to fill in the current indices.

All twelve spatial directions follow this pattern with the appropriate center's displacement vector and exponent. There are patterns that can be noticed in this relation:

- **Same-electron terms** (stepping $a_x$, reading $a_x{-}1$ or $b_x{-}1$) carry the $\eta/(\zeta+\eta)$ factor and appear as a *difference* of two Boys orders.
- **Cross-electron terms** (stepping $a_x$, reading $c_x{-}1$ or $d_x{-}1$) have no $m$-difference at all. They appear only at order $m{+}1$, and their denominator is $2(\zeta+\eta)$ rather than $2\zeta$.

When stepping a *ket* index ($c$ or $d$), the roles swap: $\mathbf Q - \mathbf C$ and
$\mathbf W - \mathbf Q$ replace $\mathbf P - \mathbf A$ and $\mathbf W - \mathbf P$, $\eta$ replaces
$\zeta$, and the $\zeta/(\zeta+\eta)$ factor replaces $\eta/(\zeta+\eta)$.

## The algorithm

It is essentially identical in strategy to the nuclear attraction integral. We climb in total angular momentum $L = 1, \ldots, \ell_a + \ell_b + \ell_c + \ell_d$, visiting only the index combinations whose twelve spatial indices sum to $L$, filling orders $m = 0, \ldots, (L_{\max} - L)$, and stepping the first nonzero spatial index we find. Building in increasing $L$ guarantees every dependency exists before it is read. `shift_ERI` is the 13-index version of `shift_NAI` from 11c, with the same shift and bound check scheme.

The Boys function and the angular combinations generator is built here, which is identical to those in 11c, so this can be an independent workbook.

In [2]:
import numpy as np
import psi4
from scipy.special import gamma, gammainc


def boys_function(m, x):
    """Boys function F_m(x), via the regularized lower incomplete gamma function."""
    if x < 1e-9:
        # Analytic limit; the general expression below is 0/0 at x = 0.
        return 1.0 / (2 * m + 1)

    return 0.5 * x ** (-(m + 0.5)) * gamma(m + 0.5) * gammainc(m + 0.5, x)


def angular_combinations(n):
    """Enumerate Cartesian (nx, ny, nz) triples of total angular momentum n in Psi4's component ordering."""
    coms = []
    for nx in range(n, -1, -1):
        for ny in range(n - nx, -1, -1):
            nz = n - nx - ny
            coms.append((nx, ny, nz))
    return coms

In [3]:
def get_ERI_table(A, B, C, D, expa, expb, expc, expd, AMa, AMb, AMc, AMd):
    """Obara-Saika electron repulsion integrals for one primitive quartet.

    Parameters
    ----------
    A, B, C, D : ndarray, shape (3,)
        Coordinates of the four shell centers.
    expa, expb, expc, expd : float
        The four primitive exponents.
    AMa, AMb, AMc, AMd : int
        Angular momenta of the four shells.

    Returns
    -------
    ndarray, shape (AMa+1,)*3 + (AMb+1,)*3 + (AMc+1,)*3 + (AMd+1,)*3 + (max_m+1,)
        Indexed [ax, ay, az, bx, by, bz, cx, cy, cz, dx, dy, dz, m], relative to ERI_base. 
        Only the m = 0 slice is physically meaningful.
    """
    max_m = AMa + AMb + AMc + AMd

    ERI_table = np.zeros((AMa + 1, AMa + 1, AMa + 1,
                    AMb + 1, AMb + 1, AMb + 1,
                    AMc + 1, AMc + 1, AMc + 1,
                    AMd + 1, AMd + 1, AMd + 1,
                    max_m + 1))

    def shift_ERI(shift_dict):
        """Read ERI_table at the current loop indices, shifted. Returns 0.0 if out of bounds."""
        axis_dict = {
            "ax": 0, "ay": 1, "az": 2,
            "bx": 3, "by": 4, "bz": 5,
            "cx": 6, "cy": 7, "cz": 8,
            "dx": 9, "dy": 10, "dz": 11,
            "m": 12
        }

        # Captures the current loop variables from the enclosing scope.
        curr_idx = [ax, ay, az, bx, by, bz, cx, cy, cz, dx, dy, dz, m]
        for axis, shift in shift_dict.items():
            curr_idx[axis_dict[axis]] += shift

        # A negative angular momentum index means the term does not exist.
        if any(idx < 0 for idx in curr_idx):
            return 0.0

        return ERI_table[tuple(curr_idx)]

    # Bra pair (electron 1) and ket pair (electron 2).
    zeta = expa + expb
    eta = expc + expd
    zeta_eta_sum = zeta + eta

    P = (expa * A + expb * B) / zeta
    Q = (expc * C + expd * D) / eta

    # The combined center W couples the two product Gaussians. 
    # W - P and W - Q play the role that P - C played for the nuclear attraction integral.
    W = (zeta * P + eta * Q) / zeta_eta_sum
    WP = W - P
    WQ = W - Q

    PA = P - A
    PB = P - B
    QC = Q - C
    QD = Q - D

    # 1. Seed every Boys order for the [00|00] base case.
    rho = (zeta * eta) / zeta_eta_sum
    T = rho * np.sum((P - Q) ** 2)
    for m in range(max_m + 1):
        ERI_table[0, 0, 0,
            0, 0, 0,
            0, 0, 0,
            0, 0, 0,
            m] = boys_function(m, T)

    # 2. Climb in total angular momentum L, so that every dependency is already filled before it is read.
    for L in range(1, max_m + 1):
        for ax in range(AMa + 1):
            for ay in range(AMa + 1 - ax):
                for az in range(AMa + 1 - ax - ay):
                    for bx in range(AMb + 1):
                        for by in range(AMb + 1 - bx):
                            for bz in range(AMb + 1 - bx - by):
                                for cx in range(AMc + 1):
                                    for cy in range(AMc + 1 - cx):
                                        for cz in range(AMc + 1 - cx - cy):
                                            for dx in range(AMd + 1):
                                                for dy in range(AMd + 1 - dx):
                                                    for dz in range(AMd + 1 - dx - dy):

                                                        # Only touch states on the current L shell.
                                                        if ax + ay + az + bx + by + bz + cx + cy + cz + dx + dy + dz != L:
                                                            continue

                                                        # Orders above max_m - L are not calculated, since they can never be consumed by the recursion.
                                                        for m in range(max_m - L + 1):

                                                            # Step up the first nonzero component found.
                                                            if ax > 0:
                                                                term1 = PA[0] * shift_ERI({"ax": -1})
                                                                term2 = WP[0] * shift_ERI({"ax": -1, "m": +1})
                                                                term3 = (0.5 / zeta) * (ax - 1) * (shift_ERI({"ax": -2}) - (eta / zeta_eta_sum) * shift_ERI({"ax": -2, "m": +1})) if ax > 1 else 0.0
                                                                term4 = (0.5 / zeta) * bx * (shift_ERI({"ax": -1, "bx": -1}) - (eta / zeta_eta_sum) * shift_ERI({"ax": -1, "bx": -1, "m": +1})) if bx > 0 else 0.0
                                                                term5 = (0.5 / zeta_eta_sum) * cx * shift_ERI({"ax": -1, "cx": -1, "m": +1}) if cx > 0 else 0.0
                                                                term6 = (0.5 / zeta_eta_sum) * dx * shift_ERI({"ax": -1, "dx": -1, "m": +1}) if dx > 0 else 0.0
                                                                ERI_table[ax, ay, az, bx, by, bz, cx, cy, cz, dx, dy, dz, m] = term1 + term2 + term3 + term4 + term5 + term6

                                                            elif ay > 0:
                                                                term1 = PA[1] * shift_ERI({"ay": -1})
                                                                term2 = WP[1] * shift_ERI({"ay": -1, "m": +1})
                                                                term3 = (0.5 / zeta) * (ay - 1) * (shift_ERI({"ay": -2}) - (eta / zeta_eta_sum) * shift_ERI({"ay": -2, "m": +1})) if ay > 1 else 0.0
                                                                term4 = (0.5 / zeta) * by * (shift_ERI({"ay": -1, "by": -1}) - (eta / zeta_eta_sum) * shift_ERI({"ay": -1, "by": -1, "m": +1})) if by > 0 else 0.0
                                                                term5 = (0.5 / zeta_eta_sum) * cy * shift_ERI({"ay": -1, "cy": -1, "m": +1}) if cy > 0 else 0.0
                                                                term6 = (0.5 / zeta_eta_sum) * dy * shift_ERI({"ay": -1, "dy": -1, "m": +1}) if dy > 0 else 0.0
                                                                ERI_table[ax, ay, az, bx, by, bz, cx, cy, cz, dx, dy, dz, m] = term1 + term2 + term3 + term4 + term5 + term6

                                                            elif az > 0:
                                                                term1 = PA[2] * shift_ERI({"az": -1})
                                                                term2 = WP[2] * shift_ERI({"az": -1, "m": +1})
                                                                term3 = (0.5 / zeta) * (az - 1) * (shift_ERI({"az": -2}) - (eta / zeta_eta_sum) * shift_ERI({"az": -2, "m": +1})) if az > 1 else 0.0
                                                                term4 = (0.5 / zeta) * bz * (shift_ERI({"az": -1, "bz": -1}) - (eta / zeta_eta_sum) * shift_ERI({"az": -1, "bz": -1, "m": +1})) if bz > 0 else 0.0
                                                                term5 = (0.5 / zeta_eta_sum) * cz * shift_ERI({"az": -1, "cz": -1, "m": +1}) if cz > 0 else 0.0
                                                                term6 = (0.5 / zeta_eta_sum) * dz * shift_ERI({"az": -1, "dz": -1, "m": +1}) if dz > 0 else 0.0
                                                                ERI_table[ax, ay, az, bx, by, bz, cx, cy, cz, dx, dy, dz, m] = term1 + term2 + term3 + term4 + term5 + term6

                                                            # Center A exhausted, stepping up center B.
                                                            elif bx > 0:
                                                                term1 = PB[0] * shift_ERI({"bx": -1})
                                                                term2 = WP[0] * shift_ERI({"bx": -1, "m": +1})
                                                                term3 = (0.5 / zeta) * ax * (shift_ERI({"ax": -1, "bx": -1}) - (eta / zeta_eta_sum) * shift_ERI({"ax": -1, "bx": -1, "m": +1})) if ax > 0 else 0.0
                                                                term4 = (0.5 / zeta) * (bx - 1) * (shift_ERI({"bx": -2}) - (eta / zeta_eta_sum) * shift_ERI({"bx": -2, "m": +1})) if bx > 1 else 0.0
                                                                term5 = (0.5 / zeta_eta_sum) * cx * shift_ERI({"bx": -1, "cx": -1, "m": +1}) if cx > 0 else 0.0
                                                                term6 = (0.5 / zeta_eta_sum) * dx * shift_ERI({"bx": -1, "dx": -1, "m": +1}) if dx > 0 else 0.0
                                                                ERI_table[ax, ay, az, bx, by, bz, cx, cy, cz, dx, dy, dz, m] = term1 + term2 + term3 + term4 + term5 + term6

                                                            elif by > 0:
                                                                term1 = PB[1] * shift_ERI({"by": -1})
                                                                term2 = WP[1] * shift_ERI({"by": -1, "m": +1})
                                                                term3 = (0.5 / zeta) * ay * (shift_ERI({"ay": -1, "by": -1}) - (eta / zeta_eta_sum) * shift_ERI({"ay": -1, "by": -1, "m": +1})) if ay > 0 else 0.0
                                                                term4 = (0.5 / zeta) * (by - 1) * (shift_ERI({"by": -2}) - (eta / zeta_eta_sum) * shift_ERI({"by": -2, "m": +1})) if by > 1 else 0.0
                                                                term5 = (0.5 / zeta_eta_sum) * cy * shift_ERI({"by": -1, "cy": -1, "m": +1}) if cy > 0 else 0.0
                                                                term6 = (0.5 / zeta_eta_sum) * dy * shift_ERI({"by": -1, "dy": -1, "m": +1}) if dy > 0 else 0.0
                                                                ERI_table[ax, ay, az, bx, by, bz, cx, cy, cz, dx, dy, dz, m] = term1 + term2 + term3 + term4 + term5 + term6

                                                            elif bz > 0:
                                                                term1 = PB[2] * shift_ERI({"bz": -1})
                                                                term2 = WP[2] * shift_ERI({"bz": -1, "m": +1})
                                                                term3 = (0.5 / zeta) * az * (shift_ERI({"az": -1, "bz": -1}) - (eta / zeta_eta_sum) * shift_ERI({"az": -1, "bz": -1, "m": +1})) if az > 0 else 0.0
                                                                term4 = (0.5 / zeta) * (bz - 1) * (shift_ERI({"bz": -2}) - (eta / zeta_eta_sum) * shift_ERI({"bz": -2, "m": +1})) if bz > 1 else 0.0
                                                                term5 = (0.5 / zeta_eta_sum) * cz * shift_ERI({"bz": -1, "cz": -1, "m": +1}) if cz > 0 else 0.0
                                                                term6 = (0.5 / zeta_eta_sum) * dz * shift_ERI({"bz": -1, "dz": -1, "m": +1}) if dz > 0 else 0.0
                                                                ERI_table[ax, ay, az, bx, by, bz, cx, cy, cz, dx, dy, dz, m] = term1 + term2 + term3 + term4 + term5 + term6

                                                            # Bra exhausted, stepping up the ket.
                                                            # QC/WQ and eta now play the roles PA/WP and zeta played above.
                                                            elif cx > 0:
                                                                term1 = QC[0] * shift_ERI({"cx": -1})
                                                                term2 = WQ[0] * shift_ERI({"cx": -1, "m": +1})
                                                                term3 = (0.5 / zeta_eta_sum) * ax * shift_ERI({"ax": -1, "cx": -1, "m": +1}) if ax > 0 else 0.0
                                                                term4 = (0.5 / zeta_eta_sum) * bx * shift_ERI({"bx": -1, "cx": -1, "m": +1}) if bx > 0 else 0.0
                                                                term5 = (0.5 / eta) * (cx - 1) * (shift_ERI({"cx": -2}) - (zeta / zeta_eta_sum) * shift_ERI({"cx": -2, "m": +1})) if cx > 1 else 0.0
                                                                term6 = (0.5 / eta) * dx * (shift_ERI({"cx": -1, "dx": -1}) - (zeta / zeta_eta_sum) * shift_ERI({"cx": -1, "dx": -1, "m": +1})) if dx > 0 else 0.0
                                                                ERI_table[ax, ay, az, bx, by, bz, cx, cy, cz, dx, dy, dz, m] = term1 + term2 + term3 + term4 + term5 + term6

                                                            elif cy > 0:
                                                                term1 = QC[1] * shift_ERI({"cy": -1})
                                                                term2 = WQ[1] * shift_ERI({"cy": -1, "m": +1})
                                                                term3 = (0.5 / zeta_eta_sum) * ay * shift_ERI({"ay": -1, "cy": -1, "m": +1}) if ay > 0 else 0.0
                                                                term4 = (0.5 / zeta_eta_sum) * by * shift_ERI({"by": -1, "cy": -1, "m": +1}) if by > 0 else 0.0
                                                                term5 = (0.5 / eta) * (cy - 1) * (shift_ERI({"cy": -2}) - (zeta / zeta_eta_sum) * shift_ERI({"cy": -2, "m": +1})) if cy > 1 else 0.0
                                                                term6 = (0.5 / eta) * dy * (shift_ERI({"cy": -1, "dy": -1}) - (zeta / zeta_eta_sum) * shift_ERI({"cy": -1, "dy": -1, "m": +1})) if dy > 0 else 0.0
                                                                ERI_table[ax, ay, az, bx, by, bz, cx, cy, cz, dx, dy, dz, m] = term1 + term2 + term3 + term4 + term5 + term6

                                                            elif cz > 0:
                                                                term1 = QC[2] * shift_ERI({"cz": -1})
                                                                term2 = WQ[2] * shift_ERI({"cz": -1, "m": +1})
                                                                term3 = (0.5 / zeta_eta_sum) * az * shift_ERI({"az": -1, "cz": -1, "m": +1}) if az > 0 else 0.0
                                                                term4 = (0.5 / zeta_eta_sum) * bz * shift_ERI({"bz": -1, "cz": -1, "m": +1}) if bz > 0 else 0.0
                                                                term5 = (0.5 / eta) * (cz - 1) * (shift_ERI({"cz": -2}) - (zeta / zeta_eta_sum) * shift_ERI({"cz": -2, "m": +1})) if cz > 1 else 0.0
                                                                term6 = (0.5 / eta) * dz * (shift_ERI({"cz": -1, "dz": -1}) - (zeta / zeta_eta_sum) * shift_ERI({"cz": -1, "dz": -1, "m": +1})) if dz > 0 else 0.0
                                                                ERI_table[ax, ay, az, bx, by, bz, cx, cy, cz, dx, dy, dz, m] = term1 + term2 + term3 + term4 + term5 + term6

                                                            # Center C exhausted, stepping up center D.
                                                            elif dx > 0:
                                                                term1 = QD[0] * shift_ERI({"dx": -1})
                                                                term2 = WQ[0] * shift_ERI({"dx": -1, "m": +1})
                                                                term3 = (0.5 / zeta_eta_sum) * ax * shift_ERI({"ax": -1, "dx": -1, "m": +1}) if ax > 0 else 0.0
                                                                term4 = (0.5 / zeta_eta_sum) * bx * shift_ERI({"bx": -1, "dx": -1, "m": +1}) if bx > 0 else 0.0
                                                                term5 = (0.5 / eta) * cx * (shift_ERI({"cx": -1, "dx": -1}) - (zeta / zeta_eta_sum) * shift_ERI({"cx": -1, "dx": -1, "m": +1})) if cx > 0 else 0.0
                                                                term6 = (0.5 / eta) * (dx - 1) * (shift_ERI({"dx": -2}) - (zeta / zeta_eta_sum) * shift_ERI({"dx": -2, "m": +1})) if dx > 1 else 0.0
                                                                ERI_table[ax, ay, az, bx, by, bz, cx, cy, cz, dx, dy, dz, m] = term1 + term2 + term3 + term4 + term5 + term6

                                                            elif dy > 0:
                                                                term1 = QD[1] * shift_ERI({"dy": -1})
                                                                term2 = WQ[1] * shift_ERI({"dy": -1, "m": +1})
                                                                term3 = (0.5 / zeta_eta_sum) * ay * shift_ERI({"ay": -1, "dy": -1, "m": +1}) if ay > 0 else 0.0
                                                                term4 = (0.5 / zeta_eta_sum) * by * shift_ERI({"by": -1, "dy": -1, "m": +1}) if by > 0 else 0.0
                                                                term5 = (0.5 / eta) * cy * (shift_ERI({"cy": -1, "dy": -1}) - (zeta / zeta_eta_sum) * shift_ERI({"cy": -1, "dy": -1, "m": +1})) if cy > 0 else 0.0
                                                                term6 = (0.5 / eta) * (dy - 1) * (shift_ERI({"dy": -2}) - (zeta / zeta_eta_sum) * shift_ERI({"dy": -2, "m": +1})) if dy > 1 else 0.0
                                                                ERI_table[ax, ay, az, bx, by, bz, cx, cy, cz, dx, dy, dz, m] = term1 + term2 + term3 + term4 + term5 + term6

                                                            elif dz > 0:
                                                                term1 = QD[2] * shift_ERI({"dz": -1})
                                                                term2 = WQ[2] * shift_ERI({"dz": -1, "m": +1})
                                                                term3 = (0.5 / zeta_eta_sum) * az * shift_ERI({"az": -1, "dz": -1, "m": +1}) if az > 0 else 0.0
                                                                term4 = (0.5 / zeta_eta_sum) * bz * shift_ERI({"bz": -1, "dz": -1, "m": +1}) if bz > 0 else 0.0
                                                                term5 = (0.5 / eta) * cz * (shift_ERI({"cz": -1, "dz": -1}) - (zeta / zeta_eta_sum) * shift_ERI({"cz": -1, "dz": -1, "m": +1})) if cz > 0 else 0.0
                                                                term6 = (0.5 / eta) * (dz - 1) * (shift_ERI({"dz": -2}) - (zeta / zeta_eta_sum) * shift_ERI({"dz": -2, "m": +1})) if dz > 1 else 0.0
                                                                ERI_table[ax, ay, az, bx, by, bz, cx, cy, cz, dx, dy, dz, m] = term1 + term2 + term3 + term4 + term5 + term6

    return ERI_table

## Eight-fold permutational symmetry

Naively the ERI tensor has $N^4$ elements, but for real orbitals only $\sim N^4/8$ of them are distinct. Three independent symmetries generate the group of eight:

\begin{equation}
(\mu\nu|\lambda\sigma) = (\nu\mu|\lambda\sigma)
\tag{bra permutation}
\end{equation}

\begin{equation}
(\mu\nu|\lambda\sigma) = (\mu\nu|\sigma\lambda)
\tag{ket permutation}
\end{equation}

\begin{equation}
(\mu\nu|\lambda\sigma) = (\lambda\sigma|\mu\nu)
\tag{bra-ket exchange}
\end{equation}

This 8-fold symmetry can be exploited inside the shell loop. A shell quartet is computed only if it is **canonical**, and non-canonical quartets are filled
by transposing a block that has already been computed. Three checks, applied in order:

| Condition | Already-computed block | Transpose |
|---|---|---|
| `d < c` | `(a, b, d, c)` | `(0, 1, 3, 2)` |
| `b < a` | `(b, a, c, d)` | `(1, 0, 2, 3)` |
| `(a, b) > (c, d)` | `(c, d, a, b)` | `(2, 3, 0, 1)` |

The ordering of the checks matters. Each one is only allowed to reference a block that the shell loops have already visited, and since `d` is the inner most loop (sweep fastest) and `a` is the outer most loop (sweep slowest), that means fixing the ket first, then the bra, then the bra-ket ordering. Only quartets satisfying $a \leq b$, $c \leq d$, and $(a,b) \leq (c,d)$ (canonical) reach the primitive loop and do real work. The transpose tuple is just the symmetry relation written in NumPy's axis notation.

## Setting up the molecule and basis set

Same setup as 11c: Cartesian Gaussians via `puream 0`, and contraction coefficients taken directly from
Psi4 so that normalization is already handled.

In [4]:
mol = psi4.geometry("""
                        O
                        H 1 1.1
                        H 1 1.1 2 104
                        symmetry c1
                        """)

psi4.set_options({'basis':        'sto-3g',
                  'scf_type':         'pk',
                  'puream':              0,
                  'e_convergence':    1e-8,
                  'd_convergence':    1e-8})

wfn = psi4.core.Wavefunction.build(mol, psi4.core.get_global_option('basis'), quiet=True)
basis = wfn.basisset()
mints = psi4.core.MintsHelper(basis)

n_ao = basis.nao()
print(f"{basis.nshell()} shells, {n_ao} Cartesian AOs")

5 shells, 7 Cartesian AOs


## Assembling the ERI tensor

The loop structure mirrors 11c, but over 4 centers instead of two:

1. **Shell quartets**, with the three symmetry checks above.
2. **Primitive quartets**, with pre-computation of variables independent of angular momentum indices
3. **Angular components**, with all combinations for all four centers.

In [5]:
ERI = np.zeros((n_ao, n_ao, n_ao, n_ao))

# Level 1: loop over shell quartets.
for a in range(basis.nshell()):
    for b in range(basis.nshell()):
        for c in range(basis.nshell()):
            for d in range(basis.nshell()):

                ashell = basis.shell(a)
                bshell = basis.shell(b)
                cshell = basis.shell(c)
                dshell = basis.shell(d)

                AMa = ashell.am
                AMb = bshell.am
                AMc = cshell.am
                AMd = dshell.am

                # Basis function index where each shell begins.
                a_idx = ashell.function_index
                b_idx = bshell.function_index
                c_idx = cshell.function_index
                d_idx = dshell.function_index

                a_coms = angular_combinations(AMa)
                b_coms = angular_combinations(AMb)
                c_coms = angular_combinations(AMc)
                d_coms = angular_combinations(AMd)

                n_coma = len(a_coms)
                n_comb = len(b_coms)
                n_comc = len(c_coms)
                n_comd = len(d_coms)

                # 8-fold symmetry: fill non-canonical quartets by transposing a block already computed
                if d < c:
                    ERI[a_idx:a_idx + n_coma, b_idx:b_idx + n_comb, c_idx:c_idx + n_comc, d_idx:d_idx + n_comd] = \
                        ERI[a_idx:a_idx + n_coma, b_idx:b_idx + n_comb, d_idx:d_idx + n_comd, c_idx:c_idx + n_comc].transpose((0, 1, 3, 2))
                    continue

                elif b < a:
                    ERI[a_idx:a_idx + n_coma, b_idx:b_idx + n_comb, c_idx:c_idx + n_comc, d_idx:d_idx + n_comd] = \
                        ERI[b_idx:b_idx + n_comb, a_idx:a_idx + n_coma, c_idx:c_idx + n_comc, d_idx:d_idx + n_comd].transpose((1, 0, 2, 3))
                    continue

                elif (a, b) > (c, d):
                    ERI[a_idx:a_idx + n_coma, b_idx:b_idx + n_comb, c_idx:c_idx + n_comc, d_idx:d_idx + n_comd] = \
                        ERI[c_idx:c_idx + n_comc, d_idx:d_idx + n_comd, a_idx:a_idx + n_coma, b_idx:b_idx + n_comb].transpose((2, 3, 0, 1))
                    continue

                # shell.ncenter gives the atom index; mol.x()/y()/z() give its coordinates.
                A = np.array([mol.x(ashell.ncenter), mol.y(ashell.ncenter), mol.z(ashell.ncenter)])
                B = np.array([mol.x(bshell.ncenter), mol.y(bshell.ncenter), mol.z(bshell.ncenter)])
                C = np.array([mol.x(cshell.ncenter), mol.y(cshell.ncenter), mol.z(cshell.ncenter)])
                D = np.array([mol.x(dshell.ncenter), mol.y(dshell.ncenter), mol.z(dshell.ncenter)])

                # Level 2: loop over primitive quartets.
                for a_prim in range(ashell.nprimitive):
                    for b_prim in range(bshell.nprimitive):
                        for c_prim in range(cshell.nprimitive):
                            for d_prim in range(dshell.nprimitive):

                                expa = ashell.exp(a_prim)
                                expb = bshell.exp(b_prim)
                                expc = cshell.exp(c_prim)
                                expd = dshell.exp(d_prim)

                                # Already carries primitive and contraction normalization.
                                coefa = ashell.coef(a_prim)
                                coefb = bshell.coef(b_prim)
                                coefc = cshell.coef(c_prim)
                                coefd = dshell.coef(d_prim)

                                ERI_table = get_ERI_table(A, B, C, D,
                                                   expa, expb, expc, expd,
                                                   AMa, AMb, AMc, AMd)

                                zeta = expa + expb
                                eta = expc + expd
                                xi_ab = (expa * expb) / zeta
                                xi_cd = (expc * expd) / eta

                                S_base_ab = (np.pi / zeta) ** (3 / 2) * np.exp(-xi_ab * np.sum((A - B) ** 2))
                                S_base_cd = (np.pi / eta) ** (3 / 2) * np.exp(-xi_cd * np.sum((C - D) ** 2))

                                rho = (zeta * eta) / (zeta + eta)
                                ERI_base = 2 * (rho / np.pi) ** 0.5 * S_base_ab * S_base_cd

                                # Level 3: loop over angular components.
                                for counta, a_com in enumerate(a_coms):
                                    for countb, b_com in enumerate(b_coms):
                                        for countc, c_com in enumerate(c_coms):
                                            for countd, d_com in enumerate(d_coms):

                                                ax, ay, az = a_com
                                                bx, by, bz = b_com
                                                cx, cy, cz = c_com
                                                dx, dy, dz = d_com

                                                ERI_term = ERI_table[ax, ay, az,
                                                                   bx, by, bz,
                                                                   cx, cy, cz,
                                                                   dx, dy, dz, 0]

                                                ERI[a_idx + counta,
                                                    b_idx + countb,
                                                    c_idx + countc,
                                                    d_idx + countd] += ERI_base \
                                                                     * coefa * coefb * coefc * coefd \
                                                                     * ERI_term


## Validation

In [6]:
psi4.compare_values(np.asarray(mints.ao_eri()), ERI, 8, "Electron Repulsion Integrals")

    Electron Repulsion Integrals..........................................................PASSED    Electron Repulsion Integrals..........................................................PASSED



True

With $\mathbf S$, $\mathbf T$, $\mathbf V$, and now the ERI tensor in hand, every input to the Roothaan-Hall equations has been built from Gaussian exponents and contraction coefficients alone.

## References

1. Recursion scheme:
    > [[Obara:1986](http://aip.scitation.org/doi/abs/10.1063/1.450106)] S. Obara and A. Saika, *J. Chem. Phys.* **84**, 3963 (1986)

2. Boys function, original definition:
    > [[Boys:1950:542](http://dx.doi.org/10.1098/rspa.1950.0036)] S. F. Boys, *Proc. R. Soc. Lond. A* **200**, 542 (1950)

3. Integral algorithms and the incomplete-gamma form of the Boys function:
    > [[Helgaker:2000](https://onlinelibrary.wiley.com/doi/book/10.1002/9781119019572)] T. Helgaker, P. Jorgensen, and J. Olsen, *Molecular Electronic-Structure Theory*, John Wiley & Sons Inc, 2000